In [4]:
import json
import sys
import re
from rich import print as rp
from collections import Counter
from pathlib import Path
from datetime import datetime

nb_dir = Path.cwd()
project_root = nb_dir.parent.parent
sys.path.insert(0, str(project_root))

from scripts.text_matching import normalise_text,remove_diacritics

In [5]:
uids_to_fix_file = Path(project_root / "scripts/notebooks/uids_2_fix.json")
with open(uids_to_fix_file, "r") as f:
   uids = json.load(f)

rp(uids[:10])

[
    {
        'uid_parts': ['heinz', 'mohr', 'gerd'],
        'unified_id': 'heinz_mohr_gerd',
        'person_id': 727,
        'display_name': 'Heinz-Mohr, Gerd',
        'family_name': 'Heinz-Mohr',
        'given_names': 'Gerd',
        'name_prefix': None,
        'name_particles': None,
        'name_suffix': None,
        'single_name': None,
        'is_organisation': False
    },
    {
        'uid_parts': ['bukowska', 'grosse', 'ewa'],
        'unified_id': 'bukowska_grosse_ewa',
        'person_id': 2917,
        'display_name': 'Bukowska-Grosse, Ewa',
        'family_name': 'Bukowska-Grosse',
        'given_names': 'Ewa',
        'name_prefix': None,
        'name_particles': None,
        'name_suffix': None,
        'single_name': None,
        'is_organisation': False
    },
    {
        'uid_parts': ['fischer', 'dieskau', 'dietrich'],
        'unified_id': 'fischer_dieskau_dietrich',
        'person_id': 5002,
        'display_name': 'Dietrich Fischer-Dieskau',
        'family_name': 'Fischer-Dieskau',
        'given_names': 'Dietrich',
        'name_prefix': None,
        'name_particles': None,
        'name_suffix': None,
        'single_name': None,
        'is_organisation': False
    },
    {
        'uid_parts': ['schmidt', 'dengler', 'wendelin'],
        'unified_id': 'schmidt_dengler_wendelin',
        'person_id': 6921,
        'display_name': 'Schmidt-Dengler, Wendelin',
        'family_name': 'Schmidt-Dengler',
        'given_names': 'Wendelin',
        'name_prefix': None,
        'name_particles': None,
        'name_suffix': None,
        'single_name': None,
        'is_organisation': False
    },
    {
        'uid_parts': ['rot'],
        'unified_id': 'rot',
        'person_id': 5602,
        'display_name': 'Rot',
        'family_name': 'Rot',
        'given_names': 'D.',
        'name_prefix': None,
        'name_particles': None,
        'name_suffix': None,
        'single_name': None,
        'is_organisation': False
    },
    {
        'uid_parts': ['kaas'],
        'unified_id': 'kaas',
        'person_id': 1268,
        'display_name': 'Kaas',
        'family_name': 'Kaas',
        'given_names': 'Harald',
        'name_prefix': None,
        'name_particles': None,
        'name_suffix': None,
        'single_name': None,
        'is_organisation': False
    },
    {
        'uid_parts': ['engel', 'janosi', 'friedrich'],
        'unified_id': 'engel_janosi_friedrich',
        'person_id': 5265,
        'display_name': 'ENGEL-JANOSI, Friedrich',
        'family_name': 'Engel-Janosi',
        'given_names': 'Friedrich',
        'name_prefix': None,
        'name_particles': None,
        'name_suffix': None,
        'single_name': None,
        'is_organisation': False
    },
    {
        'uid_parts': ['oelscher', 'obermaier', 'hans', 'p'],
        'unified_id': 'oelscher_obermaier_hans_p',
        'person_id': 42,
        'display_name': 'Oelscher-Obermaier, Hans-Peter',
        'family_name': 'Oelscher-Obermaier',
        'given_names': 'Hans-Peter',
        'name_prefix': None,
        'name_particles': None,
        'name_suffix': None,
        'single_name': None,
        'is_organisation': False
    },
    {
        'uid_parts': ['ricker', 'abderhalden', 'judith'],
        'unified_id': 'ricker_abderhalden_judith',
        'person_id': 5558,
        'display_name': 'Ricker-Abderhalden, Judith',
        'family_name': 'Ricker-Abderhalden',
        'given_names': 'Judith',
        'name_prefix': None,
        'name_particles': None,
        'name_suffix': None,
        'single_name': None,
        'is_organisation': False
    },
    {
        'uid_parts': ['goeppert', 'frank', 'herma', 'c'],
        'unified_id': 'goeppert_frank_herma_c',
        'person_id': 7467,
        'display_name': 'Goeppert-Frank, Herma C.',
        'family_name': 'Goeppert-Frank',
        'given_names': 'Herma C.',
        'name_prefix': None,
        'name_particles': None,
        'name_suffix': Non

In [6]:
for person in uids:
    family_norm = normalise_text(person["family_name"])
    given_norm = normalise_text(person["given_names"]) or ""
    #                                                   ^^^^^ 3 entries have None given

    given_tokens = given_norm.replace(".", " ").split()

    if not given_tokens:
        unified_id_base = family_norm
    elif len(given_tokens) == 1:
        unified_id_base = family_norm + "_" + given_norm.replace(".", "")
    else:
        first_full = given_tokens[0]
        initials = "_".join(t[0] for t in given_tokens[1:])
        unified_id_base = family_norm + "_" + first_full + "_" + initials

    person["unified_id"] = remove_diacritics(unified_id_base)

# which person_ids does each unified_id map to?
uid_to_person_ids = {}
for person in uids:
    uid_to_person_ids.setdefault(person["unified_id"], set()).add(person["person_id"])

# person_ids whose new unified_id already exists on a DIFFERENT person in `people`
# (found by running collision_check.sql) -> hold back for manual merge
db_collisions = {2603}   # berg_alban already on person_id 2442

# clean 1:1 -> goes into the UPDATE ; anything else -> inspect
id_updates = []      # (unified_id, person_id)
conflicts = {}       # unified_id -> entries

seen_person_ids = set()
for person in uids:
    uid = person["unified_id"]
    pid = person["person_id"]

    if len(uid_to_person_ids[uid]) > 1 or pid in db_collisions:
        conflicts.setdefault(uid, []).append(person)
    elif pid not in seen_person_ids:
        id_updates.append((uid, pid))
        seen_person_ids.add(pid)

rp(f"clean updates: {len(id_updates)}")
rp(f"conflicts (one uid -> several person_ids, or already in people): {len(conflicts)}")

# one UPDATE statement to paste into the Neon SQL editor
values_rows = ",\n".join(
    f"    ('{uid.replace(chr(39), chr(39) + chr(39))}', {pid})"
    for uid, pid in id_updates
)
update_sql = (
    "UPDATE people AS p\n"
    "SET unified_id = v.unified_id\n"
    "FROM (VALUES\n"
    f"{values_rows}\n"
    ") AS v(unified_id, person_id)\n"
    "WHERE p.person_id = v.person_id;\n"
)

with open("update_uids.sql", "w") as f:
    f.write(update_sql)
with open("conflicts.json", "w") as f:
    json.dump(conflicts, f, ensure_ascii=False, indent=2)

clean updates: 169

conflicts (one uid -> several person_ids, or already in people): 1